# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list all record sets (by their `@id`), the fields within each record set, and the fields' associated columns.

In [ ]:
# Inspect available record sets, fields, and columns
record_sets = metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record set: {rs['@id']} (name: {rs['name']})")
    fields = rs['fields']
    print(f"  Fields ({len(fields)}):")
    for field in fields:
        # Field is a dict with field metadata
        print(f"    - Field: {field['@id']} (name: {field.get('name', '')})")
        # Show column mapping if exists
        col = field.get('column')
        if col:
            print(f"      Maps to column: {col}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For this demonstration, we select the **first** record set. (Replace with the desired record set if needed.)

In [ ]:
# Collect all record set @ids
all_record_set_ids = [record_set['@id'] for record_set in metadata.record_sets]

print(f"Record set IDs: {all_record_set_ids}")

dataframes = {}
for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"---\nColumns in record set '{record_set_id}':\n{df.columns.tolist()}\n")

# Select a record set for EDA
selected_record_set_id = all_record_set_ids[0]
print(f"Using record set: {selected_record_set_id}")
display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# --- EDA ---
# First, print all column names, infer numeric fields
df = dataframes[selected_record_set_id]
print("Columns in selected DataFrame:", df.columns.tolist())

# Attempt to detect a numeric column (e.g., 'Coverage', 'MW', etc.)
possible_numeric = [col for col in df.columns if df[col].dtype in ['int64','float64'] or pd.api.types.is_numeric_dtype(df[col])]

# If all columns are object, try to convert some likely candidates by name
if not possible_numeric:
    for col in df.columns:
        if any(x in col.lower() for x in ['coverage','count','mw','abundance','number','intensity','percent','peptide','mod']):
            # Try convert
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if pd.api.types.is_numeric_dtype(df[col]):
                possible_numeric.append(col)
if not possible_numeric:
    print("No numeric field detected.")
else:
    print(f"Candidate numeric fields: {possible_numeric}")

# Choose a numeric field (customize as needed, here we take the first one)
numeric_field = possible_numeric[0] if possible_numeric else None

if numeric_field:
    print(f"Using numeric field for EDA: {numeric_field}")
    # Filter on a threshold: e.g., values > threshold (set adaptively)
    threshold = df[numeric_field].mean()  # or a fixed value
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} records\n")

    # Z-score normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - df[numeric_field].mean()) / df[numeric_field].std()
    print(f"First 5 records with normalized {numeric_field}:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to find a group field (e.g. sample, condition, type, etc.)
    group_candidates = [col for col in df.columns if col != numeric_field and df[col].nunique() < (len(df)//5) and df[col].dtype == object]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"Grouping by field: {group_field}\n")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print("No numeric field found, EDA is limited without numeric data.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.title(f"Distribution of {numeric_field}")
    plt.show()

    # If group_field found above, plot boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available to visualize.")

## 6. Conclusion
In this notebook, we have loaded the FAIR^2 dataset by URL and explored its record sets, field metadata, and extracted the records for analysis using the `mlcroissant` library. We demonstrated how to dynamically discover numeric fields, perform simple EDA (filtering and normalization), and visualize distributions from the tabular data.

Refer to the dataset's Croissant schema and documentation for additional domain-specific analyses or mapping of `@id` fields to specific variables.